# Multi-Hierarchical RL System - Part 2: Super Agent

## Train Super Agent to Ensemble Base Agents

The Super Agent learns:
- When to trust Technical vs Sentiment agent
- How to dynamically blend their predictions
- Based on recent performance, agreement, and market conditions

**Goal:** Improve beyond best base agent

In [1]:
# Imports
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

# RL libraries
from stable_baselines3 import PPO, SAC, A2C
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback
import torch as th

# Our modules
from super_agent_enviroment import SuperAgentEnv

# Plotting setup
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Config
DATA_DIR = 'data_super_agent'
MODELS_DIR = Path('models_super_agent')
MODELS_DIR.mkdir(exist_ok=True)

SEED = 42
np.random.seed(SEED)
th.manual_seed(SEED)

print("✓ Imports complete")

✓ Imports complete


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## 1. Load Base Agent Predictions

Load the predictions generated by Technical and Sentiment agents.

In [2]:
# Load metadata
with open(Path(DATA_DIR) / 'metadata.json', 'r') as f:
    metadata = json.load(f)

tickers = metadata['tickers']
print("Tickers:", tickers)
print("\nBase Models:")
print("  Technical:", metadata['base_models']['technical']['algorithm'])
print("  Sentiment:", metadata['base_models']['sentiment']['algorithm'])

# Load predictions
train_pred = pd.read_csv(Path(DATA_DIR) / 'base_predictions_train.csv', index_col=0, parse_dates=True)
val_pred = pd.read_csv(Path(DATA_DIR) / 'base_predictions_val.csv', index_col=0, parse_dates=True)
test_pred = pd.read_csv(Path(DATA_DIR) / 'base_predictions_test.csv', index_col=0, parse_dates=True)

print(f"\nPrediction Data:")
print(f"  Train: {train_pred.shape}")
print(f"  Val: {val_pred.shape}")
print(f"  Test: {test_pred.shape}")
print(f"\nFeatures: {list(train_pred.columns)}")

# Load returns for reward calculation
base_data_dir = metadata['data_source']
train_returns = pd.read_csv(Path(base_data_dir) / 'returns_train.csv', index_col=0, parse_dates=True)
val_returns = pd.read_csv(Path(base_data_dir) / 'returns_val.csv', index_col=0, parse_dates=True)
test_returns = pd.read_csv(Path(base_data_dir) / 'returns_test.csv', index_col=0, parse_dates=True)

FileNotFoundError: [Errno 2] No such file or directory: 'data_super_agent/metadata.json'

## 2. Analyze Base Agent Performance

Before training Super Agent, let's see how well the base agents perform.

In [ ]:
# Calculate base agent performance on test set
test_tech_returns = test_pred['tech_return'].values
test_sent_returns = test_pred['sent_return'].values

tech_sharpe = (test_tech_returns.mean() / test_tech_returns.std()) * np.sqrt(52)
sent_sharpe = (test_sent_returns.mean() / test_sent_returns.std()) * np.sqrt(52)

print("Base Agent Performance (Test Set):")
print(f"  Technical Sharpe: {tech_sharpe:.3f}")
print(f"  Sentiment Sharpe: {sent_sharpe:.3f}")
print(f"  Best Base: {max(tech_sharpe, sent_sharpe):.3f}")

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Cumulative returns
axes[0, 0].plot(np.cumsum(test_tech_returns), label='Technical', linewidth=2)
axes[0, 0].plot(np.cumsum(test_sent_returns), label='Sentiment', linewidth=2)
axes[0, 0].set_title('Cumulative Returns (Test Set)')
axes[0, 0].set_ylabel('Cumulative Log Return')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Return distribution
axes[0, 1].hist(test_tech_returns, bins=30, alpha=0.5, label='Technical', density=True)
axes[0, 1].hist(test_sent_returns, bins=30, alpha=0.5, label='Sentiment', density=True)
axes[0, 1].set_title('Return Distribution')
axes[0, 1].set_xlabel('Weekly Return')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Rolling Sharpe
window = 12
tech_rolling_sharpe = pd.Series(test_tech_returns).rolling(window).apply(
    lambda x: (x.mean() / x.std()) * np.sqrt(52) if x.std() > 0 else 0
)
sent_rolling_sharpe = pd.Series(test_sent_returns).rolling(window).apply(
    lambda x: (x.mean() / x.std()) * np.sqrt(52) if x.std() > 0 else 0
)

axes[1, 0].plot(tech_rolling_sharpe, label='Technical', linewidth=2)
axes[1, 0].plot(sent_rolling_sharpe, label='Sentiment', linewidth=2)
axes[1, 0].axhline(0, color='black', linestyle='--', alpha=0.3)
axes[1, 0].set_title(f'Rolling {window}-Week Sharpe')
axes[1, 0].set_ylabel('Sharpe Ratio')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Agreement over time
axes[1, 1].plot(test_pred['agreement'], label='Cosine Similarity', linewidth=2)
axes[1, 1].plot(test_pred['weight_correlation'], label='Weight Correlation', linewidth=2)
axes[1, 1].axhline(0, color='black', linestyle='--', alpha=0.3)
axes[1, 1].set_title('Agent Agreement Over Time')
axes[1, 1].set_ylabel('Agreement Metric')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Base agent analysis complete")

## 3. Training Configuration

In [ ]:
# Training configuration
TRAIN_CONFIG = {
    'total_steps': 50_000,       # Fewer steps (less data than Part 1)
    'eval_freq': 2_000,          # Evaluate more frequently
    'patience': 5,               # Early stopping patience
    'learning_rate': 3e-4,
    'gamma': 0.99,
}

# Modes to test
MODES = ['ensemble', 'selection']  # Ensemble vs binary selection
ALGORITHMS = ['PPO', 'SAC']        # Test PPO and SAC

print("Super Agent Training Configuration:")
for k, v in TRAIN_CONFIG.items():
    print(f"  {k}: {v}")
print(f"\nModes: {MODES}")
print(f"Algorithms: {ALGORITHMS}")

## 4. Training Functions

In [ ]:
# Validation callback
class SuperAgentValidationCallback(BaseCallback):
    def __init__(self, val_env, eval_freq, patience, save_path, verbose=1):
        super().__init__(verbose)
        self.val_env = val_env
        self.eval_freq = eval_freq
        self.patience = patience
        self.save_path = save_path
        self.best_improvement = -np.inf
        self.no_improve = 0
    
    def _on_step(self):
        if self.eval_freq <= 0 or (self.n_calls % self.eval_freq) != 0:
            return True
        
        # Evaluate
        results = self.val_env.run_full_pass(self.model)
        
        improvement = results['improvement']
        
        if self.verbose:
            print(f"[Step {self.num_timesteps:,}] Val Sharpe: {results['sharpe']:.3f} "
                  f"(Tech: {results['tech_sharpe']:.3f}, Sent: {results['sent_sharpe']:.3f}) "
                  f"Improvement: {improvement:+.3f}")
        
        if improvement > self.best_improvement + 1e-6:
            self.best_improvement = improvement
            self.no_improve = 0
            
            if self.save_path:
                self.model.save(self.save_path)
                if self.verbose:
                    print(f"  ✓ New best model saved (improvement: {improvement:+.3f})")
        else:
            self.no_improve += 1
            if self.no_improve >= self.patience:
                if self.verbose:
                    print(f"  ⛔ Early stopping")
                return False
        
        return True


def train_super_agent(mode, algorithm, config, verbose=True):
    """
    Train Super Agent.
    
    Parameters
    ----------
    mode : str
        'ensemble' or 'selection'
    algorithm : str
        'PPO' or 'SAC'
    config : dict
        Training config
    verbose : bool
        Print progress
    """
    if verbose:
        print(f"\n{'='*70}")
        print(f"Training Super Agent: {mode.upper()} mode with {algorithm}")
        print(f"{'='*70}")
    
    # Create environments
    train_env = SuperAgentEnv(
        train_pred, train_returns, tickers,
        mode=mode, lookback_window=8
    )
    val_env = SuperAgentEnv(
        val_pred, val_returns, tickers,
        mode=mode, lookback_window=8
    )
    test_env = SuperAgentEnv(
        test_pred, test_returns, tickers,
        mode=mode, lookback_window=8
    )
    
    # Wrap for SB3
    vec_env = DummyVecEnv([lambda: train_env])
    
    # Create model
    model_kwargs = {
        'policy': 'MlpPolicy',
        'env': vec_env,
        'learning_rate': config['learning_rate'],
        'gamma': config['gamma'],
        'seed': SEED,
        'verbose': 0 if not verbose else 1,
    }
    
    if algorithm == 'PPO':
        model = PPO(
            **model_kwargs,
            n_steps=512,
            batch_size=128,
            policy_kwargs=dict(activation_fn=th.nn.ReLU, net_arch=[128, 128])
        )
    elif algorithm == 'SAC':
        model = SAC(
            **model_kwargs,
            buffer_size=50_000,
            batch_size=256,
            policy_kwargs=dict(
                activation_fn=th.nn.ReLU,
                net_arch=dict(pi=[128, 128], qf=[128, 128])
            )
        )
    else:
        raise ValueError(f"Unknown algorithm: {algorithm}")
    
    # Training callback
    save_path = MODELS_DIR / f"best_super_{mode}_{algorithm.lower()}.zip"
    callback = SuperAgentValidationCallback(
        val_env=val_env,
        eval_freq=config['eval_freq'],
        patience=config['patience'],
        save_path=str(save_path),
        verbose=1 if verbose else 0
    )
    
    # Train
    model.learn(
        total_timesteps=config['total_steps'],
        callback=callback
    )
    
    # Load best model
    if save_path.exists():
        if algorithm == 'PPO':
            model = PPO.load(str(save_path), env=vec_env)
        elif algorithm == 'SAC':
            model = SAC.load(str(save_path), env=vec_env)
    
    # Evaluate on all splits
    train_results = train_env.run_full_pass(model)
    val_results = val_env.run_full_pass(model)
    test_results = test_env.run_full_pass(model)
    
    if verbose:
        print(f"\nResults:")
        print(f"  Train - Super: {train_results['sharpe']:.3f}, "
              f"Tech: {train_results['tech_sharpe']:.3f}, "
              f"Sent: {train_results['sent_sharpe']:.3f}, "
              f"Improvement: {train_results['improvement']:+.3f}")
        print(f"  Val   - Super: {val_results['sharpe']:.3f}, "
              f"Tech: {val_results['tech_sharpe']:.3f}, "
              f"Sent: {val_results['sent_sharpe']:.3f}, "
              f"Improvement: {val_results['improvement']:+.3f}")
        print(f"  Test  - Super: {test_results['sharpe']:.3f}, "
              f"Tech: {test_results['tech_sharpe']:.3f}, "
              f"Sent: {test_results['sent_sharpe']:.3f}, "
              f"Improvement: {test_results['improvement']:+.3f}")
    
    return {
        'mode': mode,
        'algorithm': algorithm,
        'train': train_results,
        'val': val_results,
        'test': test_results,
        'model_path': str(save_path)
    }

print("✓ Training functions defined")

## 5. Train Super Agents

Train both ensemble and selection modes with different algorithms.

In [ ]:
print("\n" + "="*70)
print("TRAINING SUPER AGENTS")
print("="*70)

all_results = []

for mode in MODES:
    for algo in ALGORITHMS:
        result = train_super_agent(mode, algo, TRAIN_CONFIG, verbose=True)
        all_results.append(result)

print("\n" + "="*70)
print("✓ TRAINING COMPLETE")
print("="*70)

## 6. Compare Results

In [ ]:
# Create comparison DataFrame
comparison_data = []

for result in all_results:
    comparison_data.append({
        'Mode': result['mode'],
        'Algorithm': result['algorithm'],
        'Train Sharpe': result['train']['sharpe'],
        'Val Sharpe': result['val']['sharpe'],
        'Test Sharpe': result['test']['sharpe'],
        'Test Improvement': result['test']['improvement'],
        'Test Tech': result['test']['tech_sharpe'],
        'Test Sent': result['test']['sent_sharpe'],
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('Test Sharpe', ascending=False)

print("\nSuper Agent Performance Comparison:")
print("="*70)
print(comparison_df.to_string(index=False))
print("="*70)

# Find best
best_result = all_results[comparison_df.index[0]]
print(f"\n🏆 Best Super Agent: {best_result['mode']} + {best_result['algorithm']}")
print(f"   Test Sharpe: {best_result['test']['sharpe']:.3f}")
print(f"   Improvement: {best_result['test']['improvement']:+.3f}")

## 7. Visualize Best Super Agent

In [ ]:
# Load best model
best_mode = best_result['mode']
best_algo = best_result['algorithm']

if best_algo == 'PPO':
    best_model = PPO.load(best_result['model_path'])
else:
    best_model = SAC.load(best_result['model_path'])

# Create test env and run
test_env = SuperAgentEnv(
    test_pred, test_returns, tickers,
    mode=best_mode, lookback_window=8
)

results = test_env.run_full_pass(best_model)

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Cumulative returns comparison
axes[0, 0].plot(np.cumsum(results['returns']), label='Super Agent', linewidth=2.5)
axes[0, 0].plot(np.cumsum(results['tech_returns']), label='Technical', linewidth=2, alpha=0.7)
axes[0, 0].plot(np.cumsum(results['sent_returns']), label='Sentiment', linewidth=2, alpha=0.7)
axes[0, 0].set_title('Cumulative Returns (Test Set)')
axes[0, 0].set_ylabel('Cumulative Log Return')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Blending weights over time (if ensemble mode)
if best_mode == 'ensemble':
    axes[0, 1].plot(results['blends'], linewidth=2)
    axes[0, 1].axhline(0.5, color='black', linestyle='--', alpha=0.3, label='Equal Weight')
    axes[0, 1].set_title('Blending Weights Over Time')
    axes[0, 1].set_ylabel('Weight (0=Sentiment, 1=Technical)')
    axes[0, 1].set_ylim(-0.1, 1.1)
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)
else:
    # Selection over time
    axes[0, 1].plot(results['blends'], linewidth=2)
    axes[0, 1].set_title('Agent Selection Over Time')
    axes[0, 1].set_ylabel('Selection (0=Sentiment, 1=Technical)')
    axes[0, 1].set_ylim(-0.1, 1.1)
    axes[0, 1].grid(alpha=0.3)

# Rolling Sharpe comparison
window = 12
super_rolling = pd.Series(results['returns']).rolling(window).apply(
    lambda x: (x.mean() / x.std()) * np.sqrt(52) if x.std() > 0 else 0
)
tech_rolling = pd.Series(results['tech_returns']).rolling(window).apply(
    lambda x: (x.mean() / x.std()) * np.sqrt(52) if x.std() > 0 else 0
)
sent_rolling = pd.Series(results['sent_returns']).rolling(window).apply(
    lambda x: (x.mean() / x.std()) * np.sqrt(52) if x.std() > 0 else 0
)

axes[1, 0].plot(super_rolling, label='Super Agent', linewidth=2.5)
axes[1, 0].plot(tech_rolling, label='Technical', linewidth=2, alpha=0.7)
axes[1, 0].plot(sent_rolling, label='Sentiment', linewidth=2, alpha=0.7)
axes[1, 0].axhline(0, color='black', linestyle='--', alpha=0.3)
axes[1, 0].set_title(f'Rolling {window}-Week Sharpe')
axes[1, 0].set_ylabel('Sharpe Ratio')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Performance summary
metrics_data = [
    ['Super Agent', results['sharpe'], results['returns'].mean() * 52, results['returns'].std() * np.sqrt(52)],
    ['Technical', results['tech_sharpe'], results['tech_returns'].mean() * 52, results['tech_returns'].std() * np.sqrt(52)],
    ['Sentiment', results['sent_sharpe'], results['sent_returns'].mean() * 52, results['sent_returns'].std() * np.sqrt(52)],
]
metrics_df = pd.DataFrame(metrics_data, columns=['Agent', 'Sharpe', 'Ann. Return', 'Ann. Vol'])

x = np.arange(len(metrics_df))
width = 0.25
axes[1, 1].bar(x - width, metrics_df['Sharpe'], width, label='Sharpe', alpha=0.8)
axes[1, 1].bar(x, metrics_df['Ann. Return']*100, width, label='Ann. Return (%)', alpha=0.8)
axes[1, 1].bar(x + width, metrics_df['Ann. Vol']*100, width, label='Ann. Vol (%)', alpha=0.8)
axes[1, 1].set_ylabel('Value')
axes[1, 1].set_title('Performance Comparison')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(metrics_df['Agent'])
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Visualization complete")

## 8. Save Best Model Configuration

In [ ]:
# Save configuration
super_agent_config = {
    'best_super_agent': {
        'mode': best_result['mode'],
        'algorithm': best_result['algorithm'],
        'model_path': best_result['model_path'],
        'test_sharpe': float(best_result['test']['sharpe']),
        'test_improvement': float(best_result['test']['improvement']),
    },
    'base_agents': metadata['base_models'],
    'all_results': comparison_df.to_dict('records'),
    'training_config': TRAIN_CONFIG,
    'created_at': pd.Timestamp.now().isoformat()
}

config_path = MODELS_DIR / 'best_super_agent.json'
with open(config_path, 'w') as f:
    json.dump(super_agent_config, f, indent=2)

print(f"✓ Super agent configuration saved to {config_path}")
print("\nBest Super Agent:")
print(json.dumps(super_agent_config['best_super_agent'], indent=2))

## 9. Summary & Next Steps

In [ ]:
print("\n" + "="*70)
print("PART 2 TRAINING COMPLETE")
print("="*70)

print("\n✅ Trained Super Agents:")
for result in all_results:
    print(f"  {result['mode']} + {result['algorithm']}: "
          f"Test Sharpe {result['test']['sharpe']:.3f} "
          f"(improvement: {result['test']['improvement']:+.3f})")

print(f"\n🏆 Best Super Agent: {best_result['mode']} + {best_result['algorithm']}")
print(f"   Test Sharpe: {best_result['test']['sharpe']:.3f}")
print(f"   Improvement over best base: {best_result['test']['improvement']:+.3f}")

if best_result['test']['improvement'] > 0:
    print("\n✨ SUCCESS! Super Agent improves over base agents!")
else:
    print("\n⚠️  Super Agent didn't improve over base agents.")
    print("   This can happen with limited data or when base agents are already optimal.")
    print("   Consider: more training steps, different architectures, or ensemble manually.")

print("\n📁 Saved Models:")
print(f"  Configuration: {config_path}")
print(f"  Best model: {best_result['model_path']}")

print("\n🎯 Next Steps (Part 3 - Meta Agent):")
print("  1. Add macro indicators (VIX, rates, credit spreads)")
print("  2. Create Meta Agent that adjusts Super Agent decisions")
print("  3. Train end-to-end hierarchical system")
print("  4. Deploy for live trading!")

print("\n" + "="*70)
print("Ready for Part 3!")
print("="*70)